# World Model Evaluation

Pipeline (top-to-bottom): each cell is one explicit step.

**Sections**
1. Setup — load model + dataset, compute baselines
2. Predictive Quality — teacher forcing + autoregressive rollout
3. Recovery — fit linear + MLP probes, evaluate
4. Rollout Consistency — observation/position drift, trajectory coherence
5. Counterfactual Controllability — probe-steered edits

In [ ]:
import sys
sys.path.insert(0, "..")
sys.path.insert(0, ".")

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import HTML, display

import helpers.nb_viz as nb_viz
import pim.eval as eval
from pim.extractors import (
    LinearExtractor, MLPExtractor, ProbeSpec, StateDefinition,
    identity_mse, hungarian_mse,
)
import pim.figures as figs
from pim.simulator.dataset import load_sample
from pim.simulator.viz import save_animation
from pim.world_models import load_checkpoint, load_dataset, make_test_loader

import importlib
importlib.reload(nb_viz)

## Config

In [ ]:
CHECKPOINT_PATH = "../runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
DATA_DIR        = "../datasets/4_fixed_refl_inview"
DEVICE          = "cuda"
BATCH_SIZE      = 512
NUM_WORKERS     = 6

N_OBJ            = 2          # objects to probe
USE_HUNGARIAN    = False      # set True for permutation-invariant matching

N_CONTEXT_PRED   = 10         # warm-up for AR rollout in prediction
N_CONTEXT_ROLL   = 20         # warm-up for fixed-window rollouts (Section 4)
N_ROLLOUT        = 20         # rollout length for Section 4
COHERENCE_N_EVAL = 500        # samples used for coherence + position drift
N_VIZ_ROLL       = 3          # 3-panel waterfalls per probe in Section 4

CTRL_N_ROLLOUT   = 15         # rollout length post-edit (Section 5)
N_VIZ_CTRL       = 3          # samples to visualise for controllability

# Which samples get drawn in the per-sample waterfall / trajectory viz.
# VIZ_SEED = None  -> first N samples (deterministic, original behaviour).
# VIZ_SEED = <int> -> a reproducible random subset chosen with that seed.
VIZ_SEED         = 0
N_VIZ_PRED       = 4          # actual-vs-predicted waterfalls in Section 2


def pick_viz_indices(n_available, n_viz, seed=VIZ_SEED):
    """Choose which sample indices to visualise.

    seed=None returns the first ``n_viz`` indices (original behaviour);
    otherwise returns a reproducible random subset (sorted) of that size.
    """
    n_viz = min(n_viz, n_available)
    if seed is None:
        return list(range(n_viz))
    rng = np.random.default_rng(seed)
    return sorted(rng.choice(n_available, size=n_viz, replace=False).tolist())

---
## 1 — Setup

In [ ]:
model, ckpt_info = load_checkpoint(CHECKPOINT_PATH, device=DEVICE)
bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits

test_loader = make_test_loader(test, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

obs_baselines = eval.compute_obs_baselines(test.obs, test.clean_obs, test.obs_noise_std)
pos_baselines = eval.compute_pos_baselines(test.positions, test.position_noise_std)

print(f"Model    : {ckpt_info.run_name}")
print(f"Epoch    : {ckpt_info.epoch}   val_loss={ckpt_info.val_loss:.6f}")
print(f"Dataset  : {test.n_samples:,} samples  T={test.T_frames}  R={test.obs_res}")

In [ ]:
# Training curves + dataset waterfall overview
display(figs.plot_training_curves(
    ckpt_info.metrics_history, best_epoch=ckpt_info.epoch,
    run_name=ckpt_info.run_name, show_components=False, log_x=False,
))
display(figs.plot_training_curves(
    ckpt_info.metrics_history, best_epoch=ckpt_info.epoch,
    run_name=ckpt_info.run_name, show_components=False, log_x=True,
))
display(figs.plot_dataset_overview(test.h5_path, n_samples=8))
plt.close("all")

---
## 2 — Predictive Quality

In [ ]:
# Teacher forcing (single pass over the loader) + autoregressive rollout (per sample)
preds_tf, states_tf = eval.teacher_force(model, test_loader, device=DEVICE)
preds_ar = eval.autoregressive_rollouts(model, test.obs, n_context=N_CONTEXT_PRED, device=DEVICE)

prediction_metrics = eval.eval_single_step(test.obs, preds_tf)
horizon_mse_noisy  = eval.eval_horizon_mse(test.obs,       preds_ar, N_CONTEXT_PRED)
horizon_mse_clean  = eval.eval_horizon_mse(test.clean_obs, preds_ar, N_CONTEXT_PRED)
context_lengths, mse_by_ctx_noisy = eval.eval_mse_by_context(model, test_loader, device=DEVICE)
mse_by_ctx_clean = ((preds_tf - test.clean_obs[:, 1:, :]) ** 2).mean(axis=(0, 2))

print(f"Next-step MSE: {prediction_metrics.mean_mse:.6f} ± {prediction_metrics.std_mse:.6f}")
print(f"Horizon MSE  step 1={horizon_mse_noisy[0]:.4f}  "
      f"step {len(horizon_mse_noisy)}={horizon_mse_noisy[-1]:.4f}")

In [ ]:
display(figs.plot_mse_by_context(
    context_lengths, mse_by_ctx_noisy, mse_by_ctx_clean,
    n_context_warmup=N_CONTEXT_PRED, baselines=obs_baselines,
))
display(figs.plot_horizon_rmse(
    horizon_mse_noisy, horizon_mse_clean,
    n_context=N_CONTEXT_PRED, baselines=obs_baselines,
))
plt.close("all")

In [ ]:
# Waterfall pairs — actual vs AR predicted (dark simulator aesthetic)
viz_idx_pred = pick_viz_indices(test.n_samples, N_VIZ_PRED, VIZ_SEED)
for i in viz_idx_pred:
    scene, obs_depth, obs_id, obs_intensity = load_sample(test.h5_path, i)
    nb_viz.plot_waterfall_pair(
        obs_depth, obs_id, obs_intensity, scene,
        preds_ar[i], N_CONTEXT_PRED,
        title=f"Sample {i}  —  actual vs predicted (warm-up={N_CONTEXT_PRED})",
        dark=True,
    )
    plt.show()

---
## 3 — Recovery

In [ ]:
# Define the state we want to recover, the GT alignment, and the probes.
state_def = StateDefinition(
    name="positions", state_shape=(N_OBJ, 2),
    extract_fn=lambda batch: batch["positions"],
)
env_states_tf = test.positions[:, :-1, :N_OBJ, :]           # align to states_tf
vis_mask_tf   = test.is_visible[:, :-1, :N_OBJ].all(axis=2)
loss_fn = hungarian_mse if USE_HUNGARIAN else identity_mse

# Probes: add/remove freely; hyperparameters live in the constructor.
probes = [
    ProbeSpec(
        name="linear",
        probe=LinearExtractor(model.hidden_size, state_def, use_lstsq=True),
        marker="x", color_idx=0, linestyle="-",
    ),
    ProbeSpec(
        name="MLP",
        probe=MLPExtractor(model.hidden_size, state_def, mlp_hidden=256, n_epochs=30, lr=5e-3),
        marker="o", color_idx=1, linestyle="--",
    ),
]

In [ ]:
train_losses = eval.fit_probes(
    probes, states_tf, env_states_tf,
    mask=vis_mask_tf, loss_fn=loss_fn, device=DEVICE,
)
recovery_metrics = eval.eval_recovery_multi(
    probes, states_tf, env_states_tf,
    mask=vis_mask_tf, use_hungarian=USE_HUNGARIAN, device=DEVICE,
)
for name, m in recovery_metrics.items():
    print(f"  {name:8s}  train_loss={train_losses[name]:.6f}  overall_mse={m.overall_mse:.6f}")

In [ ]:
display(figs.plot_recovery_bars(recovery_metrics, probes, n_obj=N_OBJ))
display(figs.plot_recovery_by_context(recovery_metrics, probes, baselines=pos_baselines))
plt.close("all")

In [ ]:
# Per-sample trajectory viz: decode the first 3 samples and plot
N_TRAJ_VIZ = 3
decoded_tf = eval.decode_states_multi(probes, states_tf[:N_TRAJ_VIZ], device=DEVICE)
import h5py
with h5py.File(test.h5_path, "r") as f:
    colors_first = f["colors"][:N_TRAJ_VIZ, :N_OBJ]

for i in range(N_TRAJ_VIZ):
    fig = figs.plot_recovery_trajectory(
        positions_gt=test.positions[i, :-1, :N_OBJ],
        decoded_per_probe={p.name: decoded_tf[p.name][i] for p in probes},
        probes=probes,
        scene_colors=colors_first[i],
        vis_mask=vis_mask_tf[i],
        sample_idx=i,
        title_suffix="teacher forcing",
    )
    display(fig); plt.close(fig)

---
## 4 — Rollout Consistency

In [ ]:
# Fixed-window rollouts on the first COHERENCE_N_EVAL samples
h_ctx, h_roll, obs_roll = eval.collect_rollouts(
    model, test.obs[:COHERENCE_N_EVAL],
    n_context=N_CONTEXT_ROLL, n_rollout=N_ROLLOUT, device=DEVICE,
)
decoded_roll = eval.decode_states_multi(probes, h_roll, device=DEVICE)

In [ ]:
drift_mse_noisy = eval.eval_observation_drift(test.obs[:COHERENCE_N_EVAL],       obs_roll, N_CONTEXT_ROLL)
drift_mse_clean = eval.eval_observation_drift(test.clean_obs[:COHERENCE_N_EVAL], obs_roll, N_CONTEXT_ROLL)

position_drift_per_probe = {
    p.name: eval.eval_position_drift(decoded_roll[p.name], test.positions[:COHERENCE_N_EVAL], N_CONTEXT_ROLL)
    for p in probes
}

# Coherence (smoothness): GT and each probe, per-sample distribution + summary
gt_window = test.positions[:COHERENCE_N_EVAL, N_CONTEXT_ROLL:N_CONTEXT_ROLL + N_ROLLOUT, :N_OBJ]
gt_coherence_scores = eval.per_sample_coherence(gt_window)
probe_coherence_scores = {p.name: eval.per_sample_coherence(decoded_roll[p.name]) for p in probes}
trajectory_coherence  = {p.name: eval.eval_trajectory_coherence(decoded_roll[p.name]) for p in probes}

print(f"  GT       smoothness mean = {gt_coherence_scores.mean():.4f}")
for p in probes:
    cm = trajectory_coherence[p.name]
    print(f"  {p.name:8s} smoothness mean = {cm.mean_score:.4f}   jump ratio = {cm.mean_jump_ratio:.2f}")

In [ ]:
display(figs.plot_observation_drift(
    drift_mse_noisy, drift_mse_clean,
    n_context=N_CONTEXT_ROLL, n_rollout=N_ROLLOUT, baselines=obs_baselines,
))
display(figs.plot_position_drift(
    position_drift_per_probe, probes,
    n_context=N_CONTEXT_ROLL, n_rollout=N_ROLLOUT, baselines=pos_baselines,
))
display(figs.plot_coherence_bar(gt_coherence_scores, probe_coherence_scores, probes))
display(figs.plot_coherence_distribution(gt_coherence_scores, probe_coherence_scores, probes))
plt.close("all")

In [ ]:
# Per-sample rollout trajectory + 3-panel waterfall (seeded sample selection)
viz_idx_roll = pick_viz_indices(COHERENCE_N_EVAL, N_VIZ_ROLL, VIZ_SEED)
for i in viz_idx_roll:
    scene_i, *_ = load_sample(test.h5_path, i)
    decoded_i = {p.name: decoded_roll[p.name][i] for p in probes}
    sample_scores = {"GT": float(gt_coherence_scores[i])}
    sample_scores.update({p.name: float(probe_coherence_scores[p.name][i]) for p in probes})

    fig = figs.plot_rollout_trajectory(
        positions_gt=test.positions[i, :, :N_OBJ],
        decoded_per_probe=decoded_i, probes=probes,
        scene_colors=scene_i.colors,
        sample_idx=i, n_context=N_CONTEXT_ROLL, n_rollout=N_ROLLOUT,
        sample_scores=sample_scores,
    )
    display(fig); plt.close(fig)

    score_str = "  ".join(f"{k}={v:.3f}" for k, v in sample_scores.items())
    fig = figs.plot_rollout_3panel(
        test_h5_path=test.h5_path,
        positions_gt=test.positions[i, :, :N_OBJ],
        obs_rollout=obs_roll[i],
        decoded_per_probe=decoded_i, probes=probes,
        sample_idx=i, n_context=N_CONTEXT_ROLL, n_rollout=N_ROLLOUT,
        suptitle=f"Sample {i}  —  rollout smoothness  {score_str}",
    )
    display(fig); plt.close(fig)

---
## 5 — Counterfactual Controllability

In [ ]:
# Pick the linear probe — it's the only one with a usable pseudoinverse.
linear_probe_spec = next(p for p in probes if isinstance(p.probe, LinearExtractor))

# Warm up each edits sample to its edit_frame; collect h at the edit.
N_CTRL = min(500, edits.n_samples)

# Choose which samples to visualise (seeded). Passing these into warm_up_to_edit
# captures the pre-edit context for exactly these samples, so the per-sample
# viz below can show any subset — not just the first few.
viz_idx_ctrl = pick_viz_indices(N_CTRL, N_VIZ_CTRL, VIZ_SEED)
print(f"  controllability viz samples: {viz_idx_ctrl}")

warm = eval.warm_up_to_edit(
    model, edits.obs[:N_CTRL], edits.edit_frame,
    viz_indices=viz_idx_ctrl, n_ctx_show=8, device=DEVICE,
)

# Steering targets = the (flattened) GT positions at edit_frame.
targets = edits.positions[:N_CTRL, edits.edit_frame, :N_OBJ, :].reshape(N_CTRL, N_OBJ * 2)

steered   = eval.rollout_steered  (model, warm.h_at_edit, targets, linear_probe_spec.probe,
                              n_rollout=CTRL_N_ROLLOUT, device=DEVICE)
unsteered = eval.rollout_unsteered(model, warm.h_at_edit,
                              n_rollout=CTRL_N_ROLLOUT, device=DEVICE)

In [ ]:
# Gradient-steered rollout using the MLP probe.
# n_steps / lr / reg_weight are the key hyperparameters to tune.
# reg_weight=0 (default) is unconstrained; increase (e.g. 0.1) to anchor
# the edit closer to the original h if the steered trajectory looks off-manifold.
GRAD_N_STEPS    = 200
GRAD_LR         = 0.01
GRAD_REG_WEIGHT = 0.0

mlp_probe_spec = next(p for p in probes if isinstance(p.probe, MLPExtractor))

steered_grad = eval.rollout_gradient_steered(
    model, warm.h_at_edit, targets, mlp_probe_spec.probe,
    n_rollout=CTRL_N_ROLLOUT,
    n_steps=GRAD_N_STEPS, lr=GRAD_LR, reg_weight=GRAD_REG_WEIGHT,
    device=DEVICE,
)
print(f"  gradient injection err = {steered_grad.injection_error:.6f}")

In [ ]:
gt_obs       = edits.obs      [:N_CTRL, edits.edit_frame : edits.edit_frame + CTRL_N_ROLLOUT]
clean_gt_obs = edits.clean_obs[:N_CTRL, edits.edit_frame : edits.edit_frame + CTRL_N_ROLLOUT]
gt_positions = edits.positions[:N_CTRL, edits.edit_frame : edits.edit_frame + CTRL_N_ROLLOUT, :N_OBJ, :]

ctrl_metrics  = eval.eval_controllability(steered, unsteered, gt_obs, clean_gt_obs)
ctrl_pos_rmse = eval.eval_position_controllability(steered, unsteered, gt_positions, probes, device=DEVICE)

ratio = ctrl_metrics.unsteered_mse / (ctrl_metrics.steered_mse + 1e-12)
print(f"  steered MSE   = {ctrl_metrics.steered_mse:.6f}")
print(f"  unsteered MSE = {ctrl_metrics.unsteered_mse:.6f}   ratio = {ratio:.2f}x")
print(f"  injection err = {ctrl_metrics.injection_error:.6f}")

In [ ]:
# Compare linear (pseudoinverse) vs gradient (MLP) steering.
ctrl_metrics_grad  = eval.eval_controllability(steered_grad, unsteered, gt_obs, clean_gt_obs)
ctrl_pos_rmse_grad = eval.eval_position_controllability(
    steered_grad, unsteered, gt_positions, probes, device=DEVICE,
)

ratio_lin  = ctrl_metrics.unsteered_mse      / (ctrl_metrics.steered_mse      + 1e-12)
ratio_grad = ctrl_metrics_grad.unsteered_mse / (ctrl_metrics_grad.steered_mse + 1e-12)
print(f"  linear   steered  MSE = {ctrl_metrics.steered_mse:.6f}   improvement = {ratio_lin:.2f}x")
print(f"  gradient steered  MSE = {ctrl_metrics_grad.steered_mse:.6f}   improvement = {ratio_grad:.2f}x")
print(f"  unsteered         MSE = {ctrl_metrics.unsteered_mse:.6f}")
print(f"  linear   injection err = {ctrl_metrics.injection_error:.6f}")
print(f"  gradient injection err = {ctrl_metrics_grad.injection_error:.6f}")

In [ ]:
display(figs.plot_controllability_obs(ctrl_metrics, baselines=obs_baselines))
display(figs.plot_controllability_positions(ctrl_pos_rmse, probes, baselines=pos_baselines))
plt.close("all")

In [ ]:
# Per-sample position trajectories + waterfalls for the seeded viz samples.
# warm.h_pre_edit rows correspond to warm.viz_indices (== viz_idx_ctrl) in order;
# the steered/unsteered rollout arrays are full-length, so index them by sample.
edit_frame = edits.edit_frame

pre_edit_decoded_all  = eval.decode_states_multi(probes, warm.h_pre_edit, device=DEVICE)
steered_decoded_all   = eval.decode_states_multi(probes, steered_grad.h[viz_idx_ctrl], device=DEVICE)
unsteered_decoded_all = eval.decode_states_multi(probes, unsteered.h[viz_idx_ctrl], device=DEVICE)

for k, i in enumerate([156, 421, 231]):
    pre_edit_gt = edits.positions[i, edit_frame - warm.n_ctx_show : edit_frame, :N_OBJ]
    post_edit_gt = gt_positions[i]

    fig = figs.plot_controllability_trajectory(
        pre_edit_gt=pre_edit_gt,
        pre_edit_decoded={p.name: pre_edit_decoded_all[p.name][k] for p in probes},
        post_edit_gt=post_edit_gt,
        steered_decoded={p.name: steered_decoded_all[p.name][k] for p in probes},
        probes=probes,
        scene_colors=edits.colors[i, :N_OBJ],
        sample_idx=i, edit_frame=edit_frame, n_rollout=CTRL_N_ROLLOUT,
        show_unsteered=True,
        unsteered_decoded={p.name: unsteered_decoded_all[p.name][k] for p in probes},
    )
    display(fig); plt.close(fig)

    fig = figs.plot_controllability_waterfalls(
        pre_edit_obs=edits.obs[i, :edit_frame],
        gt_post_obs=gt_obs[i],
        steered_obs=steered_grad.obs[i],
        unsteered_obs=unsteered.obs[i],
        sample_idx=i, edit_frame=edit_frame, n_rollout=CTRL_N_ROLLOUT,
    )
    display(fig); plt.close(fig)

---
## Appendix — animations

The GRU appendix builds two animations: GT vs decoded under teacher forcing,
and GT vs decoded with autoregressive rollout from frame T/2.

### TF animation — GT vs decoded positions

In [ ]:
ANIM_SAMPLE_IDX = 0
ANIM_INTERVAL   = 80

scene, obs_depth, obs_id, obs_intensity = load_sample(test.h5_path, ANIM_SAMPLE_IDX)

# Decode positions from teacher-forcing hidden states for this sample (linear probe)
linear_probe_obj = next(p.probe for p in probes if isinstance(p.probe, LinearExtractor))
with torch.no_grad():
    h_i = torch.from_numpy(states_tf[ANIM_SAMPLE_IDX:ANIM_SAMPLE_IDX + 1]).float().to(DEVICE)
    decoded_pos = linear_probe_obj(h_i).cpu().numpy()[0]   # (T-1, n_obj, 2)

anim = nb_viz.animate_gt_vs_predicted(
    scene, obs_depth, obs_id, obs_intensity, decoded_pos,
    interval=ANIM_INTERVAL,
    title=f"Sample {ANIM_SAMPLE_IDX}  |  GT (solid) vs decoded positions (dashed)",
    dark=True,
)
plt.close()
HTML(anim.to_jshtml())

In [ ]:
# save_animation(anim, "../outputs/gt_vs_decoded.gif", fps=12)

### AR rollout animation — GT vs decoded (3-panel)

In [ ]:
from pim.eval import autoregressive_rollout

ANIM_AR_SAMPLE_IDX = 0
ANIM_AR_N_CONTEXT  = test.T_frames // 2
ANIM_AR_INTERVAL   = 80

scene, obs_depth, obs_id, obs_intensity = load_sample(test.h5_path, ANIM_AR_SAMPLE_IDX)

# AR rollout: warm up, then roll out autoregressively; collect hidden states throughout
pred_rollout_ar, internal_states_ar = autoregressive_rollout(
    model, obs_intensity, ANIM_AR_N_CONTEXT, device=DEVICE,
)

with torch.no_grad():
    h_ar = torch.from_numpy(internal_states_ar).float().to(DEVICE)
    decoded_pos_ar = linear_probe_obj(h_ar).cpu().numpy()   # (T, n_obj, 2)

anim_ar = nb_viz.animate_ar_gt_vs_predicted(
    scene, obs_depth, obs_id, obs_intensity,
    pred_rollout_ar, decoded_pos_ar,
    n_context=ANIM_AR_N_CONTEXT,
    interval=ANIM_AR_INTERVAL,
    title=f"Sample {ANIM_AR_SAMPLE_IDX}  |  AR rollout from frame {ANIM_AR_N_CONTEXT}",
    dark=True,
)
plt.close()
HTML(anim_ar.to_jshtml())

In [ ]:
# save_animation(anim_ar, "../outputs/gt_vs_decoded_ar.gif", fps=12)